In [3]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD DATASETS
# ============================================================

OLD_FILE = "final_merged_dataset_v4.csv"
NEW_FILE = "final_merged_dataset_v5.csv"

old = pd.read_csv(OLD_FILE)
new = pd.read_csv(NEW_FILE)

print("OLD DATASET SHAPE:", old.shape)
print("NEW DATASET SHAPE:", new.shape)

OLD DATASET SHAPE: (62250, 35)
NEW DATASET SHAPE: (62250, 35)


In [4]:
def basic_audit(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nShape:")
    print(df.shape)

    print("\nDuplicate tokenIds:")
    print(df["tokenId"].duplicated().sum())

    print("\nDuplicate full rows:")
    print(df.duplicated().sum())

    print("\nMissing values:")
    missing = df.isna().sum()
    print(
        missing[missing > 0]
        .sort_values(ascending=False)
    )

    print("\nData types:")
    print(df.dtypes)

    print("\nProfitable distribution:")
    if "profitable" in df.columns:
        print(
            df["profitable"]
            .value_counts(normalize=True)
        )

    print("\nUnique pools:")
    if "pool_address" in df.columns:
        print(df["pool_address"].nunique())

    print("\nUnique token pairs:")
    if (
        "token0_address" in df.columns
        and "token1_address" in df.columns
    ):
        print(
            df[
                ["token0_address", "token1_address"]
            ]
            .drop_duplicates()
            .shape[0]
        )

In [5]:
basic_audit(old, "OLD DATASET")
basic_audit(new, "NEW DATASET")


OLD DATASET

Shape:
(62250, 35)

Duplicate tokenIds:
0

Duplicate full rows:
0

Missing values:
token0_price_pct_change_24h    55
token1_price_pct_change_24h    55
lp_prior_position_count         3
dtype: int64

Data types:
tokenId                                  int64
open_time                               object
close_time                              object
position_duration_hours                float64
range_width_normalized                 float64
pool_address                            object
token0_address                          object
token1_address                          object
fee_tier_pct                           float64
mint_amount0_adj                       float64
mint_amount1_adj                       float64
log_deposit_value_usd                  float64
token0_price_open                      float64
token1_price_open                      float64
token0_price_close                     float64
token1_price_close                     float64
deposit_value_usd      

In [6]:
old_ids = set(old["tokenId"])
new_ids = set(new["tokenId"])

common_ids = old_ids.intersection(new_ids)

old_only_ids = old_ids - new_ids
new_only_ids = new_ids - old_ids

print("Common tokenIds:", len(common_ids))
print("Only in OLD:", len(old_only_ids))
print("Only in NEW:", len(new_only_ids))

Common tokenIds: 62250
Only in OLD: 0
Only in NEW: 0


In [7]:
old_only = old[
    old["tokenId"].isin(old_only_ids)
].copy()

new_only = new[
    new["tokenId"].isin(new_only_ids)
].copy()

In [8]:
print("\nOLD-ONLY POSITIONS")
print(old_only.shape)

print("\nProfitability:")
print(
    old_only["profitable"]
    .value_counts(normalize=True)
)

print("\nDuration:")
print(
    old_only["position_duration_hours"]
    .describe()
)

print("\nRange width:")
print(
    old_only["range_width_normalized"]
    .describe()
)

print("\nFee tiers:")
print(
    old_only["fee_tier_pct"]
    .value_counts()
)

print("\nDeposit value:")
print(
    old_only["deposit_value_usd"]
    .describe()
)


OLD-ONLY POSITIONS
(0, 35)

Profitability:
Series([], Name: proportion, dtype: float64)

Duration:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: position_duration_hours, dtype: float64

Range width:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: range_width_normalized, dtype: float64

Fee tiers:
Series([], Name: count, dtype: int64)

Deposit value:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: deposit_value_usd, dtype: float64


In [9]:
common_columns = list(
    set(old.columns).intersection(set(new.columns))
)

comparison = old.merge(
    new,
    on="tokenId",
    suffixes=("_old", "_new")
)

print("Common positions:", len(comparison))

Common positions: 62250


In [10]:
columns_to_compare = [
    "profitable",
    "deposit_value_usd",
    "token0_price_open",
    "token1_price_open",
    "range_width_normalized",
    "fee_tier_pct",
    "lp_prior_position_count"
]

for col in columns_to_compare:

    old_col = f"{col}_old"
    new_col = f"{col}_new"

    if old_col in comparison.columns and new_col in comparison.columns:

        different = (
            comparison[old_col]
            .fillna(-999999)
            !=
            comparison[new_col]
            .fillna(-999999)
        )

        print(
            f"{col}: "
            f"{different.sum()} different out of "
            f"{len(comparison)}"
        )

profitable: 0 different out of 62250
deposit_value_usd: 0 different out of 62250
token0_price_open: 0 different out of 62250
token1_price_open: 0 different out of 62250
range_width_normalized: 0 different out of 62250
fee_tier_pct: 0 different out of 62250
lp_prior_position_count: 3 different out of 62250


In [11]:
profitability_mismatch = comparison[
    comparison["profitable_old"]
    !=
    comparison["profitable_new"]
]

print(
    "Profitability mismatches:",
    len(profitability_mismatch)
)

Profitability mismatches: 0


In [12]:
model_features = [

    "pool_address",
    "token0_address",
    "token1_address",

    "fee_tier_pct",
    "range_width_normalized",

    "mint_amount0_adj",
    "mint_amount1_adj",

    "log_deposit_value_usd",
    "deposit_value_usd",

    "token0_price_open",
    "token1_price_open",

    "token0_is_stablecoin",
    "token1_is_stablecoin",

    "token0_price_std_24h",
    "token0_price_pct_change_24h",
    "token0_price_range_pct_24h",

    "token1_price_std_24h",
    "token1_price_pct_change_24h",
    "token1_price_range_pct_24h",

    "pre_open_avg_daily_swap_count_3d",

    "pre_open_avg_daily_token0_volume_3d",

    "range_overlaps_recent_trading",

    "lp_prior_position_count",

    "current_price_position_in_range",

    "distance_to_lower_range_boundary",
    "distance_to_upper_range_boundary",

    "pre_open_avg_daily_volume_usd_3d",

    "token0_return_volatility_24h",
    "token1_return_volatility_24h",

    "profitable"
]

In [13]:
leaky_columns = [

    "close_time",
    "token0_price_close",
    "token1_price_close",

    "held_value_usd",
    "returned_value_usd",

    "true_fees_value_usd",
    "impermanent_loss_usd",

    "position_duration_hours",

    "decrease_event_count"
]

print("LEAKAGE CHECK")

for col in leaky_columns:

    if col in model_features:
        print("LEAK FOUND:", col)

LEAKAGE CHECK


In [14]:
compare = old.merge(
    new,
    on="tokenId",
    suffixes=("_old", "_new")
)

compare[
    compare["lp_prior_position_count_old"]
    != compare["lp_prior_position_count_new"]
][
    [
        "tokenId",
        "open_time_old",
        "lp_prior_position_count_old",
        "lp_prior_position_count_new"
    ]
]

,tokenId,open_time_old,lp_prior_position_count_old,lp_prior_position_count_new
6027,54636,2021-06-20 13:35:59+00:00,NaN,0.0
7952,64991,2021-06-29 06:44:05+00:00,NaN,0.0
61301,585523,2023-10-20 17:24:11+00:00,NaN,0.0


In [15]:
# ============================================================
# CHECK HOW MISSING 24H PRICE CHANGES WERE HANDLED IN NEW
# ============================================================

# Find positions where either 24h price-change feature was missing in OLD
missing_old = old[
    old["token0_price_pct_change_24h"].isna()
    |
    old["token1_price_pct_change_24h"].isna()
]

print("Rows with missing 24h price-change values in OLD:")
print(len(missing_old))

# Get their tokenIds
missing_token_ids = missing_old["tokenId"].tolist()

# Compare OLD vs NEW for exactly those positions
comparison = (
    old[
        old["tokenId"].isin(missing_token_ids)
    ][
        [
            "tokenId",
            "open_time",
            "token0_price_pct_change_24h",
            "token1_price_pct_change_24h"
        ]
    ]
    .merge(
        new[
            new["tokenId"].isin(missing_token_ids)
        ][
            [
                "tokenId",
                "token0_price_pct_change_24h",
                "token1_price_pct_change_24h"
            ]
        ],
        on="tokenId",
        suffixes=("_old", "_new")
    )
)

print("\nComparison:")
print(comparison.head(60))

Rows with missing 24h price-change values in OLD:
55

Comparison:
    tokenId                  open_time  token0_price_pct_change_24h_old  \
0        41  2021-05-05 02:10:21+00:00                              NaN   
1       151  2021-05-05 18:05:11+00:00                              NaN   
2       161  2021-05-05 18:10:25+00:00                              NaN   
3       165  2021-05-05 18:11:24+00:00                              NaN   
4       257  2021-05-05 19:02:16+00:00                              NaN   
5       286  2021-05-05 19:08:45+00:00                              NaN   
6       305  2021-05-05 19:15:04+00:00                              NaN   
7       310  2021-05-05 19:15:27+00:00                              NaN   
8       317  2021-05-05 19:16:16+00:00                              NaN   
9       379  2021-05-05 19:31:48+00:00                              NaN   
10      426  2021-05-05 19:38:12+00:00                              NaN   
11      436  2021-05-05 19:38:55+0

In [16]:
comparison[
    comparison["token0_price_pct_change_24h_old"].isna()
][
    [
        "tokenId",
        "open_time",
        "token0_price_pct_change_24h_old",
        "token0_price_pct_change_24h_new"
    ]
].head(60)

,tokenId,open_time,token0_price_pct_change_24h_old,token0_price_pct_change_24h_new
0,41,2021-05-05 02:10:21+00:00,NaN,0.0
1,151,2021-05-05 18:05:11+00:00,NaN,0.0
2,161,2021-05-05 18:10:25+00:00,NaN,0.0
3,165,2021-05-05 18:11:24+00:00,NaN,0.0
4,257,2021-05-05 19:02:16+00:00,NaN,0.0
5,286,2021-05-05 19:08:45+00:00,NaN,0.0
6,305,2021-05-05 19:15:04+00:00,NaN,0.0
7,310,2021-05-05 19:15:27+00:00,NaN,0.0
8,317,2021-05-05 19:16:16+00:00,NaN,0.0
9,379,2021-05-05 19:31:48+00:00,NaN,0.0


In [17]:
comparison[
    comparison["token1_price_pct_change_24h_old"].isna()
][
    [
        "tokenId",
        "open_time",
        "token1_price_pct_change_24h_old",
        "token1_price_pct_change_24h_new"
    ]
].head(60)

,tokenId,open_time,token1_price_pct_change_24h_old,token1_price_pct_change_24h_new
0,41,2021-05-05 02:10:21+00:00,NaN,0.0
1,151,2021-05-05 18:05:11+00:00,NaN,0.0
2,161,2021-05-05 18:10:25+00:00,NaN,0.0
3,165,2021-05-05 18:11:24+00:00,NaN,0.0
4,257,2021-05-05 19:02:16+00:00,NaN,0.0
5,286,2021-05-05 19:08:45+00:00,NaN,0.0
6,305,2021-05-05 19:15:04+00:00,NaN,0.0
7,310,2021-05-05 19:15:27+00:00,NaN,0.0
8,317,2021-05-05 19:16:16+00:00,NaN,0.0
9,379,2021-05-05 19:31:48+00:00,NaN,0.0


In [18]:
print(
    comparison[
        comparison["token0_price_pct_change_24h_old"].isna()
    ]["token0_price_pct_change_24h_new"]
    .value_counts(dropna=False)
)

print(
    comparison[
        comparison["token1_price_pct_change_24h_old"].isna()
    ]["token1_price_pct_change_24h_new"]
    .value_counts(dropna=False)
)

token0_price_pct_change_24h_new
0.0    55
Name: count, dtype: int64
token1_price_pct_change_24h_new
0.0    55
Name: count, dtype: int64


In [19]:
final = old.copy()

# Remove positions without sufficient 24h price history
final = final.dropna(
    subset=[
        "token0_price_pct_change_24h",
        "token1_price_pct_change_24h"
    ]
)

# Positions with no earlier LP position should be 0
final["lp_prior_position_count"] = (
    final["lp_prior_position_count"].fillna(0)
)

print(final.shape)

(62195, 35)


In [20]:
final.isna().sum()

tokenId                                0
open_time                              0
close_time                             0
position_duration_hours                0
range_width_normalized                 0
pool_address                           0
token0_address                         0
token1_address                         0
fee_tier_pct                           0
mint_amount0_adj                       0
mint_amount1_adj                       0
log_deposit_value_usd                  0
token0_price_open                      0
token1_price_open                      0
token0_price_close                     0
token1_price_close                     0
deposit_value_usd                      0
held_value_usd                         0
returned_value_usd                     0
true_fees_value_usd                    0
impermanent_loss_usd                   0
token0_is_stablecoin                   0
token1_is_stablecoin                   0
token0_price_std_24h                   0
token0_price_pct

In [21]:
print(final.shape)
print(final['profitable'].value_counts(normalize=True))
print(final['tokenId'].duplicated().sum())

(62195, 35)
profitable
1    0.600064
0    0.399936
Name: proportion, dtype: float64
0


In [22]:
leaky_columns = [
    'tokenId',                    # identifier, not a predictive feature
    'open_time',                  # timestamp; keep separately if needed
    'close_time',                 # future information
    'position_duration_hours',    # only known after position closes
    'token0_price_close',         # future information
    'token1_price_close',         # future information
    'held_value_usd',             # outcome information
    'returned_value_usd',         # outcome information
    'true_fees_value_usd',        # outcome information
    'impermanent_loss_usd',       # outcome information
    'decrease_event_count'        # only known after opening
]

In [23]:
model_data = final.drop(columns=leaky_columns)

In [24]:
print(model_data.shape)
print(model_data.columns.tolist())
print(model_data.isna().sum().sum())

(62195, 24)
['range_width_normalized', 'pool_address', 'token0_address', 'token1_address', 'fee_tier_pct', 'mint_amount0_adj', 'mint_amount1_adj', 'log_deposit_value_usd', 'token0_price_open', 'token1_price_open', 'deposit_value_usd', 'token0_is_stablecoin', 'token1_is_stablecoin', 'token0_price_std_24h', 'token0_price_pct_change_24h', 'token0_price_range_pct_24h', 'token1_price_std_24h', 'token1_price_pct_change_24h', 'token1_price_range_pct_24h', 'pre_open_avg_daily_swap_count_3d', 'pre_open_avg_daily_token0_volume_3d', 'range_overlaps_recent_trading', 'lp_prior_position_count', 'profitable']
0


In [25]:
print(model_data['profitable'].value_counts(normalize=True))
print(model_data.dtypes)

profitable
1    0.600064
0    0.399936
Name: proportion, dtype: float64
range_width_normalized                 float64
pool_address                            object
token0_address                          object
token1_address                          object
fee_tier_pct                           float64
mint_amount0_adj                       float64
mint_amount1_adj                       float64
log_deposit_value_usd                  float64
token0_price_open                      float64
token1_price_open                      float64
deposit_value_usd                      float64
token0_is_stablecoin                     int64
token1_is_stablecoin                     int64
token0_price_std_24h                   float64
token0_price_pct_change_24h            float64
token0_price_range_pct_24h             float64
token1_price_std_24h                   float64
token1_price_pct_change_24h            float64
token1_price_range_pct_24h             float64
pre_open_avg_daily_swap_count_3d   

In [26]:
predictor_columns = model_data.drop(columns=['profitable']).columns.tolist()

for col in predictor_columns:
    print(col)

range_width_normalized
pool_address
token0_address
token1_address
fee_tier_pct
mint_amount0_adj
mint_amount1_adj
log_deposit_value_usd
token0_price_open
token1_price_open
deposit_value_usd
token0_is_stablecoin
token1_is_stablecoin
token0_price_std_24h
token0_price_pct_change_24h
token0_price_range_pct_24h
token1_price_std_24h
token1_price_pct_change_24h
token1_price_range_pct_24h
pre_open_avg_daily_swap_count_3d
pre_open_avg_daily_token0_volume_3d
range_overlaps_recent_trading
lp_prior_position_count


In [27]:
# Check whether lp_prior_position_count is monotonic for positions ordered by open time
audit_lp = final[['tokenId', 'open_time', 'lp_prior_position_count']].copy()

audit_lp['open_time'] = pd.to_datetime(audit_lp['open_time'], utc=True)
audit_lp = audit_lp.sort_values('open_time')

print(audit_lp.head(20))

    tokenId                 open_time  lp_prior_position_count
55     1351 2021-05-05 23:10:01+00:00                     55.0
56     1352 2021-05-05 23:10:06+00:00                     56.0
57     1390 2021-05-05 23:26:30+00:00                     57.0
58     1394 2021-05-05 23:27:10+00:00                     58.0
59     1393 2021-05-05 23:27:10+00:00                     59.0
60     1398 2021-05-05 23:28:04+00:00                     60.0
61     1404 2021-05-05 23:28:30+00:00                     61.0
62     1410 2021-05-05 23:29:02+00:00                     62.0
63     1411 2021-05-05 23:30:05+00:00                     63.0
64     1426 2021-05-05 23:33:56+00:00                     64.0
65     1427 2021-05-05 23:33:56+00:00                     65.0
66     1462 2021-05-05 23:44:01+00:00                     66.0
67     1464 2021-05-05 23:44:49+00:00                     67.0
68     1481 2021-05-05 23:47:10+00:00                     68.0
69     1485 2021-05-05 23:48:19+00:00                  

In [28]:
expected_prior_count = np.arange(len(audit_lp))

mismatch = audit_lp[
    audit_lp['lp_prior_position_count'].to_numpy() != expected_prior_count
]

print("Mismatches:", len(mismatch))
print(mismatch.head(20))

Mismatches: 62184
    tokenId                 open_time  lp_prior_position_count
55     1351 2021-05-05 23:10:01+00:00                     55.0
56     1352 2021-05-05 23:10:06+00:00                     56.0
57     1390 2021-05-05 23:26:30+00:00                     57.0
58     1394 2021-05-05 23:27:10+00:00                     58.0
59     1393 2021-05-05 23:27:10+00:00                     59.0
60     1398 2021-05-05 23:28:04+00:00                     60.0
61     1404 2021-05-05 23:28:30+00:00                     61.0
62     1410 2021-05-05 23:29:02+00:00                     62.0
63     1411 2021-05-05 23:30:05+00:00                     63.0
64     1426 2021-05-05 23:33:56+00:00                     64.0
65     1427 2021-05-05 23:33:56+00:00                     65.0
66     1462 2021-05-05 23:44:01+00:00                     66.0
67     1464 2021-05-05 23:44:49+00:00                     67.0
68     1481 2021-05-05 23:47:10+00:00                     68.0
69     1485 2021-05-05 23:48:19+00:00

In [29]:
audit_lp = final[['tokenId', 'open_time', 'lp_prior_position_count']].copy()

audit_lp['open_time'] = pd.to_datetime(audit_lp['open_time'], utc=True)
audit_lp = audit_lp.sort_values(['open_time', 'tokenId']).reset_index(drop=True)

# Difference between stored count and current row position
audit_lp['difference'] = (
    audit_lp['lp_prior_position_count']
    - audit_lp.index
)

print(audit_lp['difference'].value_counts().sort_index().head(20))

difference
-62170.0    1
-62169.0    1
-62163.0    1
-62159.0    1
-62134.0    1
-62132.0    1
-62129.0    1
-62128.0    1
-62126.0    1
-62116.0    1
-62112.0    1
-62087.0    2
-62083.0    1
-62080.0    1
-62079.0    1
-62048.0    1
-62043.0    1
-62037.0    2
-62010.0    1
-61998.0    1
Name: count, dtype: int64


In [30]:
audit_lp[['tokenId', 'open_time', 'lp_prior_position_count', 'difference']].head(20)

,tokenId,open_time,lp_prior_position_count,difference
0,1351,2021-05-05 23:10:01+00:00,55.0,55.0
1,1352,2021-05-05 23:10:06+00:00,56.0,55.0
2,1390,2021-05-05 23:26:30+00:00,57.0,55.0
3,1393,2021-05-05 23:27:10+00:00,59.0,56.0
4,1394,2021-05-05 23:27:10+00:00,58.0,54.0
5,1398,2021-05-05 23:28:04+00:00,60.0,55.0
6,1404,2021-05-05 23:28:30+00:00,61.0,55.0
7,1410,2021-05-05 23:29:02+00:00,62.0,55.0
8,1411,2021-05-05 23:30:05+00:00,63.0,55.0
9,1426,2021-05-05 23:33:56+00:00,64.0,55.0


In [31]:
model_data.to_csv("merged_cleaned_and_validated_dataset.csv")

In [33]:
model_data.columns

Index(['range_width_normalized', 'pool_address', 'token0_address',
       'token1_address', 'fee_tier_pct', 'mint_amount0_adj',
       'mint_amount1_adj', 'log_deposit_value_usd', 'token0_price_open',
       'token1_price_open', 'deposit_value_usd', 'token0_is_stablecoin',
       'token1_is_stablecoin', 'token0_price_std_24h',
       'token0_price_pct_change_24h', 'token0_price_range_pct_24h',
       'token1_price_std_24h', 'token1_price_pct_change_24h',
       'token1_price_range_pct_24h', 'pre_open_avg_daily_swap_count_3d',
       'pre_open_avg_daily_token0_volume_3d', 'range_overlaps_recent_trading',
       'lp_prior_position_count', 'profitable'],
      dtype='object')